In [1]:
# Task 1 - Install minikanren in the notebook's Python environment.
# If Task 2 cannot import kanren, remove both the # and the space at the start
# of the next line, then run this cell.
%pip install minikanren


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Task 2 - Load the family facts and define relationship rules.
import json
from pathlib import Path
from kanren import Relation, conde, facts, run, var
from kanren.constraints import neq

family = json.loads(Path("relationships.json").read_text(encoding="utf-8"))
father = Relation()
mother = Relation()

for item in family["father"]:
    facts(father, *item.items())

for item in family["mother"]:
    facts(mother, *item.items())

def parent(person, child):
    return conde([father(person, child)], [mother(person, child)])

def grandparent(person, child):
    middle = var()
    return conde([parent(person, middle), parent(middle, child)])

def sibling(first, second):
    shared_parent = var()
    return conde([parent(shared_parent, first),
                  parent(shared_parent, second), neq(first, second)])

def uncle(person, child):
    child_parent = var()
    known_child = var()
    # In this dataset, the two uncles are both listed as fathers.
    return conde([parent(child_parent, child), sibling(person, child_parent),
                  father(person, known_child), neq(person, child)])

person = var()
print("John's children:", sorted(run(0, person, father("John", person))))
print("Adam's parents:", sorted(run(0, person, parent(person, "Adam"))))
print("Megan's grandchildren:", sorted(run(0, person, grandparent("Megan", person))))


John's children: ['Adam', 'David', 'William']
Adam's parents: ['John', 'Megan']
Megan's grandchildren: ['Chris', 'Julie', 'Neil', 'Peter', 'Sophia', 'Stephanie', 'Tiffany', 'Wayne']


In [3]:
# Tasks 2.1 and 2.2 - Query William's mother and Tiffany's uncles.
print("William's mother:", sorted(run(0, person, mother(person, "William"))))
# The same uncle can be inferred more than once. List each name once.
print("Tiffany's uncles:", sorted(set(run(0, person, uncle(person, "Tiffany")))))


William's mother: ['Megan']
Tiffany's uncles: ['Adam', 'William']


In [4]:
# Task 3 - Load the coastal and adjacent-state facts.
coastal = Relation()
adjacent = Relation()

coastal_text = Path("coastal_states.txt").read_text(encoding="utf-8")
for name in coastal_text.strip().split(","):
    facts(coastal, (name.strip(),))

for line in Path("adjacent_states.txt").read_text(encoding="utf-8").splitlines():
    if not line.strip():
        continue
    names = [name.strip() for name in line.split(",")]
    for neighbour in names[1:]:
        facts(adjacent, (names[0], neighbour))

state = var()

# Use a logic variable when a state name needs to be found.
print("Is Nevada adjacent to Louisiana?",
      "Yes" if run(1, state, adjacent("Nevada", "Louisiana")) else "No")
print("Neighbours of Oregon:", sorted(run(0, state, adjacent("Oregon", state))))
print("Coastal neighbours of Mississippi:",
      sorted(run(0, state, adjacent("Mississippi", state), coastal(state))))
print("Neighbours of both Arkansas and Kentucky:",
      sorted(run(0, state, adjacent("Arkansas", state), adjacent("Kentucky", state))))
print("Neighbours of both Oklahoma and Texas:",
      sorted(run(0, state, adjacent("Oklahoma", state), adjacent("Texas", state))))
print("Coastal neighbours of both New Mexico and Louisiana:",
      sorted(run(0, state, adjacent("New Mexico", state),
                 adjacent("Louisiana", state), coastal(state))))


Is Nevada adjacent to Louisiana? No
Neighbours of Oregon: ['California', 'Idaho', 'Nevada', 'Washington']
Coastal neighbours of Mississippi: ['Alabama', 'Louisiana']
Neighbours of both Arkansas and Kentucky: ['Missouri', 'Tennessee']
Neighbours of both Oklahoma and Texas: ['Arkansas', 'New Mexico']
Coastal neighbours of both New Mexico and Louisiana: ['Texas']


In [5]:
# Tasks 3.1 and 3.2 - Filter Alabama's neighbours and list Nevada's neighbours.
print("Coastal neighbours of Alabama:",
      sorted(run(0, state, adjacent("Alabama", state), coastal(state))))
print("Neighbours of Nevada:", sorted(run(0, state, adjacent("Nevada", state))))


Coastal neighbours of Alabama: ['Florida', 'Georgia']
Neighbours of Nevada: ['Arizona', 'California', 'Idaho', 'Oregon', 'Utah']


## Task 4 - Reflective Writing

In the family tree task, I learned the difference between a fact and a rule. The answer that Megan is William's mother comes directly from the data file. To find Tiffany's uncles, the program starts with Tiffany's parents and checks which known fathers are their siblings. The query returns Adam and William.

In the geography task, I learned how to use two conditions in the same query. To find the coastal states that border Alabama, the program checks whether a state borders Alabama and is in the coastal list. The result is Florida and Georgia. The Nevada query only checks for neighbouring states, so it returns Arizona, California, Idaho, Oregon and Utah.
